# 01 — Inventario y calidad estructural del demo

Este notebook abre localmente MIMIC-IV Demo v2.2 con DuckDB. No usa PostgreSQL ni solicita credenciales. Comprueba tablas, columnas clave y relaciones necesarias para construir la cohorte.

In [ ]:
from pathlib import Path
import duckdb
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data' / 'mimic-iv-demo' / '2.2'
if not DATA_DIR.exists():
    raise FileNotFoundError('Ejecute primero 00_download_and_inspect_demo.ipynb')
connection = duckdb.connect()

In [ ]:
inventory = []
for domain in ('hosp', 'icu'):
    for path in sorted((DATA_DIR / domain).glob('*.csv.gz')):
        row_count = connection.execute(
            'SELECT count(*) FROM read_csv_auto(?)', [str(path)]
        ).fetchone()[0]
        inventory.append({
            'domain': domain,
            'table': path.name.removesuffix('.csv.gz'),
            'rows': row_count,
            'size_mib': round(path.stat().st_size / 1024 / 1024, 3),
        })
inventory = pd.DataFrame(inventory)
inventory

In [ ]:
required = {
    'hosp': {'patients', 'admissions', 'labevents', 'microbiologyevents', 'prescriptions'},
    'icu': {'icustays', 'chartevents', 'inputevents'},
}
available = {domain: set(group.table) for domain, group in inventory.groupby('domain')}
missing = {domain: sorted(tables - available.get(domain, set())) for domain, tables in required.items()}
assert not any(missing.values()), f'Tablas ausentes: {missing}'
print('Todas las tablas esenciales están disponibles.')

In [ ]:
patients = str(DATA_DIR / 'hosp' / 'patients.csv.gz')
admissions = str(DATA_DIR / 'hosp' / 'admissions.csv.gz')
icustays = str(DATA_DIR / 'icu' / 'icustays.csv.gz')
linkage = connection.execute(
    '''
    WITH p AS (SELECT * FROM read_csv_auto(?)),
         a AS (SELECT * FROM read_csv_auto(?)),
         i AS (SELECT * FROM read_csv_auto(?))
    SELECT count(*) AS icu_stays,
           count(*) FILTER (WHERE p.subject_id IS NULL) AS missing_patient_link,
           count(*) FILTER (WHERE a.hadm_id IS NULL) AS missing_admission_link,
           count(DISTINCT i.stay_id) AS unique_stay_ids
    FROM i
    LEFT JOIN p USING (subject_id)
    LEFT JOIN a USING (subject_id, hadm_id)
    ''',
    [patients, admissions, icustays],
).df()
linkage

## Criterio de salida

El paso es correcto si aparecen todas las tablas esenciales, no hay enlaces ausentes y `unique_stay_ids` coincide con `icu_stays`.

In [ ]:
connection.close()